# 04 - Orquestación con Databricks Jobs

Este notebook crea programáticamente un Databricks Job para orquestar el pipeline principal del proyecto.

El DAG implementado es:

```text
Bronze ingestion
      |
      v
Silver transformation
```

La segunda tarea depende explícitamente de la primera y solo se ejecuta cuando Bronze termina correctamente.

Se utiliza Serverless Compute para Jobs, por lo que no se fija un cluster específico en la definición de las tareas.

## 1. Configuración

Se define el catálogo que será enviado como parámetro a los notebooks del pipeline y se detecta automáticamente la raíz del proyecto a partir del directorio actual del notebook.

In [ ]:
import os

dbutils.widgets.text("catalog_name", "accidentalidad_databricks", "Catalog del proyecto")

PROY_CATALOG = dbutils.widgets.get("catalog_name")
JOB_NAME = "accidentalidad_barranquilla_pipeline"

# En Databricks Runtime moderno, el CWD de un notebook Python corresponde
# al directorio que contiene el notebook. Como este notebook vive en
# 04_Orquestacion/, el directorio padre corresponde a la raíz del proyecto.
CURRENT_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(CURRENT_DIR)

print(f"Catalog       : {PROY_CATALOG}")
print(f"Job name      : {JOB_NAME}")
print(f"Current dir   : {CURRENT_DIR}")
print(f"Project root  : {PROJECT_ROOT}")

## 2. Cliente de Databricks y resolución de notebooks

Se utiliza el Databricks SDK para Python con la autenticación del notebook actual.

La función `resolver_notebook` prueba tanto la ruta sin extensión como la ruta `.ipynb`, para soportar cómo el workspace represente los notebooks del Git Folder.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import JobSettings

w = WorkspaceClient()

def resolver_notebook(ruta_base):
    candidatos = [ruta_base, f"{ruta_base}.ipynb"]

    for candidato in candidatos:
        try:
            estado = w.workspace.get_status(candidato)
            if estado.object_type is not None:
                return candidato
        except Exception:
            pass

    raise ValueError(
        f"No se encontró el notebook. Rutas probadas: {candidatos}"
    )

BRONZE_NOTEBOOK = resolver_notebook(
    f"{PROJECT_ROOT}/01_Exploracion_y_Transformacion/01_Ingesta_Bronze"
)

SILVER_NOTEBOOK = resolver_notebook(
    f"{PROJECT_ROOT}/01_Exploracion_y_Transformacion/02_Transformacion_Silver"
)

print(f"Bronze notebook: {BRONZE_NOTEBOOK}")
print(f"Silver notebook: {SILVER_NOTEBOOK}")

## 3. Definición del Job

El Job contiene dos tareas de notebook:

1. `bronze_ingestion`: ejecuta la ingesta de los cuatro CSV hacia Bronze.
2. `silver_transformation`: depende de `bronze_ingestion` y transforma Bronze hacia Silver.

`run_if = ALL_SUCCESS` garantiza que Silver no se ejecute si la ingesta Bronze falla.

Ambas tareas reciben el mismo `catalog_name`, manteniendo parametrizado el pipeline.

In [ ]:
job_settings = JobSettings.from_dict({
    "name": JOB_NAME,
    "max_concurrent_runs": 1,
    "tasks": [
        {
            "task_key": "bronze_ingestion",
            "description": "Ingesta CSV hacia tablas Delta Bronze",
            "notebook_task": {
                "notebook_path": BRONZE_NOTEBOOK,
                "source": "WORKSPACE",
                "base_parameters": {
                    "catalog_name": PROY_CATALOG
                }
            },
        },
        {
            "task_key": "silver_transformation",
            "description": "Limpieza y transformación Bronze hacia Silver",
            "depends_on": [
                {"task_key": "bronze_ingestion"}
            ],
            "run_if": "ALL_SUCCESS",
            "notebook_task": {
                "notebook_path": SILVER_NOTEBOOK,
                "source": "WORKSPACE",
                "base_parameters": {
                    "catalog_name": PROY_CATALOG
                }
            },
        },
    ],
})

print("Definición del Job preparada correctamente.")

## 4. Creación idempotente del Job

Para evitar crear duplicados cada vez que se ejecuta este notebook, primero se busca un Job con el mismo nombre. Si ya existe, se reutiliza; si no existe, se crea.

In [ ]:
existing_job = None

for job in w.jobs.list():
    if job.settings and job.settings.name == JOB_NAME:
        existing_job = job
        break

if existing_job:
    JOB_ID = existing_job.job_id
    print(f"[OK] Job existente reutilizado: {JOB_NAME} | job_id={JOB_ID}")
else:
    created_job = w.jobs.create(**job_settings.as_shallow_dict())
    JOB_ID = created_job.job_id
    print(f"[OK] Job creado: {JOB_NAME} | job_id={JOB_ID}")

JOB_URL = f"{w.config.host}/jobs/{JOB_ID}"
print(f"Job URL: {JOB_URL}")

## 5. Verificación de la definición

Se consulta nuevamente el Job desde Databricks para dejar evidencia de sus tareas y dependencias.

In [ ]:
job_info = w.jobs.get(JOB_ID)

print(f"Job: {job_info.settings.name}")
print(f"Job ID: {JOB_ID}")
print("\nDAG:")

for task in job_info.settings.tasks:
    dependencias = (
        [dep.task_key for dep in task.depends_on]
        if task.depends_on
        else []
    )

    print(
        f"- {task.task_key} | "
        f"depends_on={dependencias or 'ninguna'} | "
        f"run_if={task.run_if or 'ALL_SUCCESS'}"
    )

## 6. Ejecución del Job

Se dispara una ejecución para demostrar que el Job es operativo. La ejecución ocurre de forma independiente al notebook actual y puede seguirse desde la URL impresa.

In [ ]:
run = w.jobs.run_now(job_id=JOB_ID)
RUN_ID = run.run_id
RUN_URL = f"{w.config.host}/jobs/{JOB_ID}/runs/{RUN_ID}"

print(f"[OK] Ejecución iniciada | run_id={RUN_ID}")
print(f"Run URL: {RUN_URL}")

## 7. Evidencia esperada

En **Jobs & Pipelines** debe observarse un Job llamado `accidentalidad_barranquilla_pipeline` con este DAG:

```text
bronze_ingestion
       |
       v
silver_transformation
```

La dependencia asegura que las transformaciones de Silver solo se ejecuten después de una ingesta Bronze exitosa.

Este patrón hace explícita la dependencia entre etapas y permite que Databricks gestione ejecución, estado, reintentos, observabilidad y futuras extensiones del pipeline.